In [142]:
import pandas as pd
import csv
from pathlib import Path

In [143]:
DATA_DIR = Path("./../../data/AusPAR")
FILE_GLOB = "*.csv"
SAVE_MERGED = True
MERGED_OUT = DATA_DIR / "AusPAR_merged.csv"

In [144]:
def into_df(input):
    return pd.read_csv(input, delimiter='~', encoding='utf-8', skipinitialspace=True, on_bad_lines='skip')

In [145]:
# inspect an example file

input_path = "./../../data/AusPAR/COGNOS_V_GEN_PRODUCT.csv"
product_df = into_df(input_path)
print(len(product_df))
product_df.columns

31942


Index(['PRODUCT_ID', 'PRODUCT_NAME', 'LICENCE_ID', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_SUPPLIED_DATE',
       'PRODUCT_CEASED_DATE', 'PRODUCT_TYPE', 'SHELF_LIFE_CONTAINER_INFO',
       'ADDITIONAL_WARNING_INFO', 'PRODUCT_CODE', 'CREATION_DATE',
       'LAST_UPDATE_DATE'],
      dtype='object')

In [146]:
product_df = pd.read_csv(
    "./../../data/AusPAR/COGNOS_V_GEN_PRODUCT.csv",
    delimiter='~',
    encoding='utf-8',
    skipinitialspace=True
)
print(len(product_df))
product_df.columns

31942


Index(['PRODUCT_ID', 'PRODUCT_NAME', 'LICENCE_ID', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_SUPPLIED_DATE',
       'PRODUCT_CEASED_DATE', 'PRODUCT_TYPE', 'SHELF_LIFE_CONTAINER_INFO',
       'ADDITIONAL_WARNING_INFO', 'PRODUCT_CODE', 'CREATION_DATE',
       'LAST_UPDATE_DATE'],
      dtype='object')

In [147]:
licence_df = pd.read_csv(
    "./../../data/AusPAR/COGNOS_V_GEN_LICENCE.csv",
    delimiter='~',
    encoding='utf-8',
    skipinitialspace=True
)
print(len(licence_df))

31921


/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_65185/1268044796.py:1: DtypeWarning: Columns (19,20,24,28,37) have mixed types. Specify dtype option on import or set low_memory=False.
  licence_df = pd.read_csv(


In [148]:
# merge by LICENCE_ID
merged_df = pd.merge(
    product_df,
    licence_df[['SPONSOR_ID', 'LICENCE_ID', 'LICENCE_STATUS']],
    on='LICENCE_ID',
    how='left'
)

print(len(merged_df))

merged_df.columns

31942


Index(['PRODUCT_ID', 'PRODUCT_NAME', 'LICENCE_ID', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_SUPPLIED_DATE',
       'PRODUCT_CEASED_DATE', 'PRODUCT_TYPE', 'SHELF_LIFE_CONTAINER_INFO',
       'ADDITIONAL_WARNING_INFO', 'PRODUCT_CODE', 'CREATION_DATE',
       'LAST_UPDATE_DATE', 'SPONSOR_ID', 'LICENCE_STATUS'],
      dtype='object')

In [149]:

# component_route_df = pd.read_csv(
#     "./../../data/AusPAR/COGNOS_V_GEN_COMPONENT_ADMIN_ROUTE.csv",
#     delimiter='~',
#     encoding='utf-8',
#     skipinitialspace=True
# )
# print(len(component_route_df))


# # merge by ROUTE_OF_ADMIN_CODE
# merged_df = pd.merge(
#     merged_df,
#     component_route_df[['ROUTE_OF_ADMIN_CODE', 'COMPONENT_ID']],
#     on='COMPONENT_ID',
#     how='left'
# )

# print(len(merged_df))

# merged_df.columns

In [150]:
merged_df = merged_df[merged_df["PRODUCT_NAME"].str.strip().astype(bool)]

In [151]:
len(merged_df)

31942

In [152]:
# value counts for PRODUCT_ID
product_id_counts = merged_df['PRODUCT_ID'].value_counts()
print(product_id_counts)

PRODUCT_ID
42646     1
540187    1
545744    1
547621    1
547619    1
         ..
391258    1
391257    1
391256    1
401224    1
568681    1
Name: count, Length: 31942, dtype: int64


In [153]:
cols_to_drop = ['LAST_UPDATE_DATE', 
                'SHELF_LIFE_CONTAINER_INFO', 
                'WEIGHT_OF_DIVIDED_PREPARATION', 
                'MAX_DAILY_DOSE', 
                'MAX_DAILY_DOSE_UNIT', 
                'MAX_SINGLE_DOSE', 
                'MAX_SINGLE_DOSE_UNIT', 
                'PRODUCT_SUPPLIED_DATE',
                'DOSAGE_FORM_CODE',
                'ADDITIONAL_WARNING_INFO',
                'VISUAL_IDENTIFICATION',
                'CREATION_DATE',
                'PRODUCT_CEASED_DATE'
                ]
merged_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
merged_df.columns

Index(['PRODUCT_ID', 'PRODUCT_NAME', 'LICENCE_ID', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_TYPE', 'PRODUCT_CODE',
       'SPONSOR_ID', 'LICENCE_STATUS'],
      dtype='object')

In [154]:
empty_rows = merged_df["PRODUCT_NAME"].isna()
# drop rows where PRODUCT_NAME is NaN
merged_df = merged_df[~empty_rows]
len(merged_df)

31942

In [155]:
merged_df.to_csv(
    DATA_DIR / "AusPAR_merged_final.csv",
    # index=False,
    sep=",",
    # quoting=csv.QUOTE_MINIMAL
)
print(f"written to: {MERGED_OUT.resolve()}")

written to: /Users/shtosti/Dropbox/Projects/DrugFork/data/AusPAR/AusPAR_merged.csv
